In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from tabulate import tabulate
from fastridge import RidgeEM, RidgeLOOCV
from experiments import RealDataExperiment
from data import get_dataset

In [ ]:
tasks = [
    ('abalone',  'Rings'),
    ('airfoil',  'scaled-sound-pressure'),
    # ('automobile',  'price'),                      # ValueError
    ('concrete', 'Concrete compressive strength'),
    ('diabetes', 'target'),
    # ('facebook',    'Total Interactions'),         # ValueError
    ('forest',      'area'),
    # ('parkinsons',  'motor_UPDRS'),                # expensive
    # ('real_estate', 'Y house price of unit area'), # expensive
    ('student',     'G3'),
    ('yacht',    'Residuary_resistance'),            # much lower r2
    # ('crime', 'ViolentCrimesPerPop')               # expensive
]
names      = [t[0] for t in tasks]
targets    = [t[1] for t in tasks]
dataframes = [get_dataset(name) for name in names]

estimators = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results = RealDataExperiment(dataframes, targets, names, estimators, n_iterations=10, seed=1, verbose=False)

table = {}
for data_name, data_result in results.items():
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {est: data_result[est]['r2'] for est in data_result}
    row['Speed-Up'] = cv_time / em_time
    row['p']       = data_result['EM']['p']
    row['n_train'] = data_result['EM']['n_train']
    row['n:p']     = data_result['EM']['n_train'] / data_result['EM']['p']
    table[data_name] = row
tdf = pd.DataFrame(table)
print(tabulate(tdf.transpose().sort_values('n_train', ascending=False),
               headers='keys', tablefmt='psql', floatfmt='.2f'))

+----------+-------+----------+----------+------------+-------+-----------+--------+
|          |    EM |   CV_glm |   CV_fix |   Speed-Up |     p |   n_train |    n:p |
|----------+-------+----------+----------+------------+-------+-----------+--------|
| abalone  |  0.54 |     0.54 |     0.54 |       8.57 |  9.00 |   2923.00 | 324.78 |
| airfoil  |  0.52 |     0.52 |     0.52 |      11.01 |  5.00 |   1052.00 | 210.40 |
| concrete |  0.59 |     0.59 |     0.59 |       9.58 |  8.00 |    721.00 |  90.12 |
| student  |  0.83 |     0.83 |     0.83 |       6.39 | 41.00 |    454.00 |  11.07 |
| forest   | -0.07 |    -0.27 |    -0.09 |       2.72 | 26.30 |    361.00 |  13.73 |
| diabetes |  0.47 |     0.47 |     0.47 |       8.61 | 10.00 |    309.00 |  30.90 |
| yacht    |  0.65 |     0.65 |     0.65 |       9.99 |  6.00 |    215.00 |  35.83 |
+----------+-------+----------+----------+------------+-------+-----------+--------+


In [ ]:
# Full experiment matching legacy notebook — skip-execution in CI
tasks_full = [
    ('abalone',     'Rings'),
    ('automobile',  'price'),
    ('autompg',     'mpg'),
    ('airfoil',     'scaled-sound-pressure'),
    ('crime',       'ViolentCrimesPerPop'),
    ('concrete',    'Concrete compressive strength'),
    ('diabetes',    'target'),
    ('facebook',    'Total Interactions'),
    ('forest',      'area'),
    ('parkinsons',  'motor_UPDRS'),
    ('real_estate', 'Y house price of unit area'),
    ('student',     'G3'),
    ('yacht',       'Residuary_resistance'),
]
names_full      = [t[0] for t in tasks_full]
targets_full    = [t[1] for t in tasks_full]
dataframes_full = [get_dataset(name) for name in names_full]

estimators_full = {
    'EM':     RidgeEM(),
    'CV_glm': RidgeLOOCV(alphas=100),
    'CV_fix': RidgeLOOCV(alphas=np.logspace(-10, 10, 100, endpoint=True, base=10)),
}

results_full = RealDataExperiment(dataframes_full, targets_full, names_full,
                                  estimators_full, n_iterations=100, seed=1, verbose=False)

table_full = {}
for data_name, data_result in results_full.items():
    em_time = data_result['EM']['time']
    cv_time = (data_result['CV_glm']['time'] + data_result['CV_fix']['time']) / 2
    row = {est: data_result[est]['r2'] for est in data_result}
    row['Speed Up Ratio'] = cv_time / em_time
    row['p']       = data_result['EM']['p']
    row['n_train'] = data_result['EM']['n_train']
    row['n:p']     = data_result['EM']['n_train'] / data_result['EM']['p']
    table_full[data_name] = row
tdf_full = pd.DataFrame(table_full)
print(tabulate(tdf_full.transpose().sort_values('n_train', ascending=False),
               headers='keys', tablefmt='psql', floatfmt='.2f'))